# 환경 준비 — 실행 환경과 경로

**목적:** 로컬/Colab 환경, repository, persistent artifact root, GPU를 확인한다.  
**입력:** Graph-CLaD checkout. 공식 연구 phase 이전의 환경 준비 단계.  
**출력:** side effect 없는 preflight dictionary.  
Drive mount는 선택 셀에서만 수행하며 기존 runtime/process를 재시작하지 않는다.

## 주요 설정
`GRAPH_CLAD_PROJECT_ROOT`, `GRAPH_CLAD_ARTIFACT_ROOT`, `GRAPH_CLAD_LIBERO_ROOT`를 환경에 맞게 설정한다. 지정하지 않으면 로컬은 `outputs/`, Colab은 persistent Drive를 기본 artifact root로 사용한다.

In [ ]:
from pathlib import Path
import os, sys

candidate = Path(os.environ.get('GRAPH_CLAD_PROJECT_ROOT', Path.cwd())).resolve()
if not (candidate / 'scripts').is_dir() and Path('/content/Graph-CLaD').is_dir():
    candidate = Path('/content/Graph-CLaD')
if not (candidate / 'scripts').is_dir():
    raise FileNotFoundError('Set GRAPH_CLAD_PROJECT_ROOT to the Graph-CLaD checkout')
os.environ['GRAPH_CLAD_PROJECT_ROOT'] = str(candidate)
os.chdir(candidate)
if str(candidate) not in sys.path:
    sys.path.insert(0, str(candidate))
from scripts.research_paths import preflight, resolve_research_paths
paths = resolve_research_paths()
preflight(paths)

In [ ]:
# Colab에서 Drive가 아직 없을 때만 명시적으로 True로 바꾼다.
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from scripts.research_paths import mount_colab_drive
    mount_colab_drive()
    paths = resolve_research_paths()
preflight(paths)

In [ ]:
import importlib.util, platform
runtime = {'python': platform.python_version()}
runtime['numpy'] = importlib.util.find_spec('numpy') is not None
runtime['torch'] = importlib.util.find_spec('torch') is not None
runtime['h5py'] = importlib.util.find_spec('h5py') is not None
runtime['libero'] = importlib.util.find_spec('libero') is not None
if runtime['torch']:
    import torch
    runtime['cuda_available'] = torch.cuda.is_available()
    runtime['gpu'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
runtime

In [ ]:
required = [
    paths.project_root / 'RESEARCH_GUIDE.md',
    paths.project_root / 'scripts/phase3/run_corrected_architecture_gate.py',
    paths.project_root / 'configs/phase3_holder_action_eval_v2_corrected.json',
]
{'required_files': {str(p): p.exists() for p in required}, 'next': 'phase_0_clad_baseline_smoke.ipynb'}

## 결과 확인과 다음 phase
`scripts_exists=True`를 확인한다. Colab 학습은 `artifact_root_exists=True`, GPU 학습은 `cuda_available=True`여야 한다. 다음은 공식 `Phase 0`의 `phase_0_clad_baseline_smoke.ipynb`다.